# Cadence Vendor-Involvement Baseline

PU learning model: 120 Cadence TINs as labeled positives, 7,134 unlabeled. 38 features from RPM claims (M&R FFS), tre_membership, and fraud_scores. Score all 7,254 TINs on vendor-involvement probability.

In [5]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings("ignore")

engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

df = pd.read_sql("select * from tmp_1m.knd_rpm_cadence_baseline", engine)
df.columns = df.columns.str.lower()
print(f"Shape: {df.shape}")
print(f"Cadence: {df['is_cadence'].sum()}, Non-Cadence: {(df['is_cadence'] == 0).sum()}")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVJdb9owFP0rkfecxEmBggVUDMSK1A8EYZV4mRzHAQ%2FHDr5OA%2Fv1cwJI3UMrTbIlyz7nnuN77vDhVEjvnRsQWo1QFGDkccV0JtRuhDbJ3O8jDyxVGZVa8RE6c0AP4yHQQpZkUtm9WvFjxcF6rpAC0jyMUGUU0RQEEEULDsQysp48P5E4wIQCcGOdHLpSMhBOa29tScKwruugvgu02YUxxjjEg9ChGsg39EGi%2FFqjNNpqpuWNcnJ%2F%2BkQiCnGnkXAIp7C8Er8LdWnBVyrpBQTkMUmW%2FvJ1nSBvcvvdVCuoCm7W3LwLxjerp4sBcA6q%2Fc53O6sphV97GoDSdS7pgTNdlJV1NQN3CnOehVLvhOvUYjZC5UFkxwPbvm2Op0T%2FfnxJplTVUqXz53R3%2FvMker1tn%2Bofk5S%2BrWbAkPfzlmvc5LoAqPhCNWlad4Xjno8HftRPog5xC98F94P%2BFnkzl6ZQ1LbMm%2BXWR1AIZjTo3GolheKtyyzF3Zwy6rN%2BTP1OOsj8dMC6Ps57nbR33%2B124ihsMovRZW5Ia8SM%2F68bw%2FAj9zqALy6TxWyppWBnb65NQe3nkUVB1N6IzM9bKOEFFXKSZYYDuOik1PXUcGrdnFtTcRSOL6r%2FTvr4Lw%3D%3D&RelayState=ver%3A3-hint%3A260092392366162-ETMsDgAAAaC03gMVABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEP6lZKCnBa2dRi34ItMv2zEAAACQCmQI%2Fw

In [6]:
# feature definitions
id_cols = ["prov_tin", "sample_full_nm", "hospital_group", "tadm_prov_spec_cat",
           "provider_state", "provider_zip", "is_cadence"]

continuous_features = [
    "avg_member_age", "std_member_age", "pct_female", "pct_htn", "pct_dm", "pct_hf", "pct_invalid_dx",
    "avg_plan_tenure_months",
    "total_mbrs", "months_active", "avg_rpm_tenure_months", "single_month_churn_rate", "peak_mom_growth_pct",
    "pct_mgmt_lines", "pct_device_lines", "pct_setup_lines", "pct_99458", "proc_hhi",
    "avg_allowed_per_line", "avg_lines_per_mbr", "top_dom_pct", "weekend_pct", "avg_submission_lag",
    "distinct_srvc_npis", "mbrs_per_srvc_npi", "pct_srvc_ne_bil",
    "pct_no_prior_rel", "pct_no_treatment_mgmt", "pct_multi_practice_overlap", "max_jaccard", "pct_shared_npis",
]

binary_features = [
    "sig_ramp", "sig_no_prior", "sig_no_treat", "sig_mbr_multi_tin",
    "sig_billing_conc", "sig_churn", "sig_tin_network",
]

all_features = continuous_features + binary_features

null_counts = df[all_features].isnull().sum()
print("Features with nulls:")
print(null_counts[null_counts > 0])

Features with nulls:
std_member_age            1481
avg_plan_tenure_months       7
mbrs_per_srvc_npi          239
pct_srvc_ne_bil            354
dtype: int64


In [7]:
# impute nulls
df["std_member_age"] = df["std_member_age"].fillna(0)
df["avg_plan_tenure_months"] = df["avg_plan_tenure_months"].fillna(df["avg_plan_tenure_months"].median())
df["pct_srvc_ne_bil"] = df["pct_srvc_ne_bil"].fillna(0)
df["mbrs_per_srvc_npi"] = df["mbrs_per_srvc_npi"].fillna(0)

assert df[all_features].isnull().sum().sum() == 0, "Nulls remain after imputation"
print("Nulls cleared.")

Nulls cleared.


In [8]:
# correlation filter on continuous features -- drop one from pairs with |r| > 0.90
corr = df[continuous_features].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k = 1).astype(bool))
high_corr_pairs = [(col, upper[col].idxmax(), upper[col].max())
                   for col in upper.columns if upper[col].max() > 0.90]

print("High-correlation pairs (|r| > 0.90):")
for c1, c2, r in sorted(high_corr_pairs, key = lambda x: -x[2]):
    print(f"  {c1} <-> {c2}: {r:.3f}")

to_drop = set()
for c1, c2, r in high_corr_pairs:
    to_drop.add(c2)

print(f"\nDropping {len(to_drop)} features: {to_drop}")
model_continuous = [f for f in continuous_features if f not in to_drop]
model_features = model_continuous + binary_features
print(f"Remaining features: {len(model_features)} ({len(model_continuous)} continuous + {len(binary_features)} binary)")

High-correlation pairs (|r| > 0.90):

Dropping 0 features: set()
Remaining features: 38 (31 continuous + 7 binary)


In [9]:
# log-transform right-skewed continuous features, then scale
skew_features = ["total_mbrs", "peak_mom_growth_pct", "distinct_srvc_npis",
                 "mbrs_per_srvc_npi", "avg_lines_per_mbr", "avg_submission_lag"]
skew_to_transform = [f for f in skew_features if f in model_continuous]

X_raw = df[model_features].copy()
for f in skew_to_transform:
    X_raw[f] = np.log1p(X_raw[f].clip(lower = 0))

scaler = StandardScaler()
X_raw[model_continuous] = scaler.fit_transform(X_raw[model_continuous])

X = X_raw.values
y = df["is_cadence"].values

print(f"X shape: {X.shape}, y positives: {y.sum()}, y negatives: {(y == 0).sum()}")

X shape: (7254, 38), y positives: 120, y negatives: 7134


## Descriptive Baseline

Cadence vs population: Cohen's d, Mann-Whitney U with BH-FDR correction.

In [10]:
# descriptive stats + statistical testing on RAW (pre-transform) features
cadence = df[df["is_cadence"] == 1]
non_cadence = df[df["is_cadence"] == 0]

results = []
for f in all_features:
    c_vals = cadence[f].dropna()
    nc_vals = non_cadence[f].dropna()

    c_med, c_q1, c_q3 = c_vals.median(), c_vals.quantile(0.25), c_vals.quantile(0.75)
    nc_med, nc_q1, nc_q3 = nc_vals.median(), nc_vals.quantile(0.25), nc_vals.quantile(0.75)

    pooled_std = np.sqrt(((len(c_vals) - 1) * c_vals.std()**2 + (len(nc_vals) - 1) * nc_vals.std()**2)
                         / (len(c_vals) + len(nc_vals) - 2))
    d = (c_vals.mean() - nc_vals.mean()) / pooled_std if pooled_std > 0 else 0

    u_stat, p_val = stats.mannwhitneyu(c_vals, nc_vals, alternative = "two-sided")

    results.append({
        "feature": f,
        "cadence_median": c_med, "cadence_iqr": f"{c_q1:.3f}-{c_q3:.3f}",
        "pop_median": nc_med, "pop_iqr": f"{nc_q1:.3f}-{nc_q3:.3f}",
        "cohens_d": d, "abs_d": abs(d),
        "mw_p": p_val,
    })

baseline = (
    pd.DataFrame(results)
    .assign(
        fdr_p = lambda x: stats.false_discovery_control(x["mw_p"], method = "bh"),
        significant = lambda x: x["fdr_p"] < 0.05,
        direction = lambda x: np.where(x["cohens_d"] > 0, "Cadence higher", "Cadence lower"),
    )
    .sort_values("abs_d", ascending = False)
)

print(f"Significant features (FDR < 0.05): {baseline['significant'].sum()} of {len(baseline)}")
print()
print(baseline[["feature", "cadence_median", "pop_median", "cohens_d", "fdr_p", "significant", "direction"]]
      .head(20).to_string(index = False))

Significant features (FDR < 0.05): 23 of 38

               feature  cadence_median  pop_median  cohens_d        fdr_p  significant      direction
    distinct_srvc_npis        6.000000    1.000000  1.199190 3.248385e-34         True Cadence higher
           top_dom_pct        0.215251    0.435205 -0.750937 6.653815e-22         True  Cadence lower
      pct_device_lines        0.516191    0.325565  0.683390 4.545951e-23         True Cadence higher
        pct_mgmt_lines        0.399776    0.624857 -0.665059 2.988626e-23         True  Cadence lower
             pct_99458        0.058032    0.145832 -0.630304 3.099020e-06         True  Cadence lower
                pct_hf        0.126330    0.000000  0.613031 1.927638e-36         True Cadence higher
        pct_invalid_dx        0.012652    0.029412 -0.480811 5.005636e-03         True  Cadence lower
        std_member_age        8.224533    6.461290  0.478373 7.833110e-10         True Cadence higher
avg_plan_tenure_months       62.32452

## PU Bagging Model

K=100 iterations. Each: 120 positives + 120 sampled unlabeled. Test both LR and RF as base learners.

In [11]:
K = 100
np.random.seed(42)

pos_idx = np.where(y == 1)[0]
unl_idx = np.where(y == 0)[0]
n_pos = len(pos_idx)

lr_scores = np.zeros((K, len(y)))
rf_scores = np.zeros((K, len(y)))
lr_coefs = np.zeros((K, X.shape[1]))
rf_importances = np.zeros((K, X.shape[1]))

for k in range(K):
    neg_sample = np.random.choice(unl_idx, size = n_pos, replace = False)
    train_idx = np.concatenate([pos_idx, neg_sample])
    X_train = X[train_idx]
    y_train = np.concatenate([np.ones(n_pos), np.zeros(n_pos)])

    lr = LogisticRegression(penalty = "l2", C = 1.0, max_iter = 1000, random_state = k)
    lr.fit(X_train, y_train)
    lr_scores[k] = lr.predict_proba(X)[:, 1]
    lr_coefs[k] = lr.coef_[0]

    rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5, random_state = k)
    rf.fit(X_train, y_train)
    rf_scores[k] = rf.predict_proba(X)[:, 1]
    rf_importances[k] = rf.feature_importances_

    if (k + 1) % 25 == 0:
        print(f"  Iteration {k + 1}/{K} complete")

lr_vendor_score = lr_scores.mean(axis = 0)
rf_vendor_score = rf_scores.mean(axis = 0)
print(f"\nLR scores: mean={lr_vendor_score.mean():.4f}, std={lr_vendor_score.std():.4f}")
print(f"RF scores: mean={rf_vendor_score.mean():.4f}, std={rf_vendor_score.std():.4f}")

  Iteration 25/100 complete
  Iteration 50/100 complete
  Iteration 75/100 complete
  Iteration 100/100 complete

LR scores: mean=0.2420, std=0.2662
RF scores: mean=0.2668, std=0.1650


In [12]:
# feature importance summary
importance = (
    pd.DataFrame({
        "feature": model_features,
        "lr_coef_mean": lr_coefs.mean(axis = 0),
        "lr_coef_abs": np.abs(lr_coefs.mean(axis = 0)),
        "rf_importance": rf_importances.mean(axis = 0),
    })
    .assign(
        lr_rank = lambda x: x["lr_coef_abs"].rank(ascending = False).astype(int),
        rf_rank = lambda x: x["rf_importance"].rank(ascending = False).astype(int),
    )
    .sort_values("rf_importance", ascending = False)
)

print("Top 15 features by RF importance:")
print(importance[["feature", "lr_coef_mean", "lr_rank", "rf_importance", "rf_rank"]]
      .head(15).to_string(index = False))

Top 15 features by RF importance:
              feature  lr_coef_mean  lr_rank  rf_importance  rf_rank
       pct_mgmt_lines     -0.498644        9       0.128931        1
          top_dom_pct     -0.693033        6       0.109386        2
               pct_hf      0.280032       24       0.095724        3
     pct_device_lines      0.407230       15       0.084014        4
            pct_99458     -0.376630       19       0.065071        5
   avg_submission_lag      0.239251       25       0.061358        6
   distinct_srvc_npis      0.782981        5       0.060031        7
pct_no_treatment_mgmt     -0.346665       22       0.048083        8
      pct_setup_lines      0.230196       26       0.035579        9
             proc_hhi      0.802442        4       0.032911       10
          weekend_pct      0.469395       11       0.029199       11
       pct_invalid_dx     -1.100962        2       0.024202       12
       std_member_age      0.064329       36       0.020848       13


## Evaluation

Held-out positive recall, precision at top-k, score distribution, stability.

In [13]:
# held-out positive recall: 5-fold CV
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
holdout_recalls_lr = []
holdout_recalls_rf = []

for fold, (train_pos_rel, test_pos_rel) in enumerate(skf.split(pos_idx, np.ones(n_pos))):
    train_pos = pos_idx[train_pos_rel]
    test_pos = pos_idx[test_pos_rel]
    n_train_pos = len(train_pos)

    fold_lr_scores = np.zeros((50, len(y)))
    fold_rf_scores = np.zeros((50, len(y)))

    for k in range(50):
        neg_sample = np.random.choice(unl_idx, size = n_train_pos, replace = False)
        train_idx = np.concatenate([train_pos, neg_sample])
        X_train = X[train_idx]
        y_train = np.concatenate([np.ones(n_train_pos), np.zeros(n_train_pos)])

        lr = LogisticRegression(penalty = "l2", C = 1.0, max_iter = 1000, random_state = fold * 100 + k)
        lr.fit(X_train, y_train)
        fold_lr_scores[k] = lr.predict_proba(X)[:, 1]

        rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5,
                                    random_state = fold * 100 + k)
        rf.fit(X_train, y_train)
        fold_rf_scores[k] = rf.predict_proba(X)[:, 1]

    fold_lr_avg = fold_lr_scores.mean(axis = 0)
    fold_rf_avg = fold_rf_scores.mean(axis = 0)

    top_200_lr = np.argsort(-fold_lr_avg)[:200]
    top_200_rf = np.argsort(-fold_rf_avg)[:200]
    recall_lr = len(set(test_pos) & set(top_200_lr)) / len(test_pos)
    recall_rf = len(set(test_pos) & set(top_200_rf)) / len(test_pos)
    holdout_recalls_lr.append(recall_lr)
    holdout_recalls_rf.append(recall_rf)
    print(f"  Fold {fold + 1}: LR recall@200={recall_lr:.3f}, RF recall@200={recall_rf:.3f}")

print(f"\nLR mean recall@200: {np.mean(holdout_recalls_lr):.3f} +/- {np.std(holdout_recalls_lr):.3f}")
print(f"RF mean recall@200: {np.mean(holdout_recalls_rf):.3f} +/- {np.std(holdout_recalls_rf):.3f}")

  Fold 1: LR recall@200=0.583, RF recall@200=0.708
  Fold 2: LR recall@200=0.458, RF recall@200=0.750
  Fold 3: LR recall@200=0.292, RF recall@200=0.667
  Fold 4: LR recall@200=0.542, RF recall@200=0.750
  Fold 5: LR recall@200=0.292, RF recall@200=0.625

LR mean recall@200: 0.433 +/- 0.122
RF mean recall@200: 0.700 +/- 0.049


In [14]:
# precision at top-k + score distribution
for label, scores in [("LR", lr_vendor_score), ("RF", rf_vendor_score)]:
    print(f"\n{label} precision at top-k:")
    for k in [100, 200, 300, 500]:
        top_k = np.argsort(-scores)[:k]
        cadence_in_top = y[top_k].sum()
        precision = cadence_in_top / k
        print(f"  Top {k}: {int(cadence_in_top)}/{k} Cadence ({precision:.1%}), base rate = {y.mean():.1%}")

for label, scores in [("LR", lr_vendor_score), ("RF", rf_vendor_score)]:
    cadence_scores = scores[y == 1]
    non_cadence_scores = scores[y == 0]
    print(f"\n{label} score distribution:")
    print(f"  Cadence:     median={np.median(cadence_scores):.4f}, "
          f"Q1={np.percentile(cadence_scores, 25):.4f}, Q3={np.percentile(cadence_scores, 75):.4f}")
    print(f"  Non-Cadence: median={np.median(non_cadence_scores):.4f}, "
          f"Q1={np.percentile(non_cadence_scores, 25):.4f}, Q3={np.percentile(non_cadence_scores, 75):.4f}")


LR precision at top-k:
  Top 100: 32/100 Cadence (32.0%), base rate = 1.7%
  Top 200: 54/200 Cadence (27.0%), base rate = 1.7%
  Top 300: 69/300 Cadence (23.0%), base rate = 1.7%
  Top 500: 87/500 Cadence (17.4%), base rate = 1.7%

RF precision at top-k:
  Top 100: 75/100 Cadence (75.0%), base rate = 1.7%
  Top 200: 91/200 Cadence (45.5%), base rate = 1.7%
  Top 300: 100/300 Cadence (33.3%), base rate = 1.7%
  Top 500: 109/500 Cadence (21.8%), base rate = 1.7%

LR score distribution:
  Cadence:     median=0.8998, Q1=0.7626, Q3=0.9526
  Non-Cadence: median=0.1248, Q1=0.0334, Q3=0.3560

RF score distribution:
  Cadence:     median=0.8973, Q1=0.6673, Q3=0.9701
  Non-Cadence: median=0.2326, Q1=0.1455, Q3=0.3415


In [15]:
# stability test
np.random.seed(999)
lr_scores_2 = np.zeros((K, len(y)))
rf_scores_2 = np.zeros((K, len(y)))

for k in range(K):
    neg_sample = np.random.choice(unl_idx, size = n_pos, replace = False)
    train_idx = np.concatenate([pos_idx, neg_sample])
    X_train = X[train_idx]
    y_train = np.concatenate([np.ones(n_pos), np.zeros(n_pos)])

    lr = LogisticRegression(penalty = "l2", C = 1.0, max_iter = 1000, random_state = 5000 + k)
    lr.fit(X_train, y_train)
    lr_scores_2[k] = lr.predict_proba(X)[:, 1]

    rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5, random_state = 5000 + k)
    rf.fit(X_train, y_train)
    rf_scores_2[k] = rf.predict_proba(X)[:, 1]

lr_vendor_score_2 = lr_scores_2.mean(axis = 0)
rf_vendor_score_2 = rf_scores_2.mean(axis = 0)

lr_rho, _ = stats.spearmanr(lr_vendor_score, lr_vendor_score_2)
rf_rho, _ = stats.spearmanr(rf_vendor_score, rf_vendor_score_2)
print(f"LR Spearman rho between two runs: {lr_rho:.4f}")
print(f"RF Spearman rho between two runs: {rf_rho:.4f}")
print(f"\nTarget: rho > 0.90. {'Both pass.' if min(lr_rho, rf_rho) > 0.90 else 'Stability concern.'}")

LR Spearman rho between two runs: 0.9946
RF Spearman rho between two runs: 0.9991

Target: rho > 0.90. Both pass.


In [16]:
# select the better model
if np.mean(holdout_recalls_rf) >= np.mean(holdout_recalls_lr):
    best_model = "RF"
    vendor_score = rf_vendor_score
    best_importance = importance.sort_values("rf_importance", ascending = False)
else:
    best_model = "LR"
    vendor_score = lr_vendor_score
    best_importance = importance.sort_values("lr_coef_abs", ascending = False)

print(f"Selected model: {best_model}")
print(f"Score range: [{vendor_score.min():.4f}, {vendor_score.max():.4f}]")

Selected model: RF
Score range: [0.0200, 0.9968]


## Score All TINs

In [17]:
# assign scores and tiers
df["vendor_score"] = vendor_score
df["vendor_pctile"] = df["vendor_score"].rank(pct = True)
df["vendor_tier"] = pd.cut(
    df["vendor_pctile"],
    bins = [0, 0.70, 0.85, 0.95, 1.0],
    labels = ["Minimal", "Low", "Medium", "High"],
    include_lowest = True,
)

top_features_list = best_importance["feature"].head(10).tolist()

def get_top_features(row, features, n = 3):
    deviations = {}
    for f in features:
        pop_med = df[f].median()
        pop_std = df[f].std()
        if pop_std > 0:
            deviations[f] = abs(row[f] - pop_med) / pop_std
        else:
            deviations[f] = 0
    top = sorted(deviations, key = deviations.get, reverse = True)[:n]
    return ", ".join(top)

df["top_contributing_features"] = df.apply(lambda r: get_top_features(r, top_features_list), axis = 1)

tier_summary = df.groupby("vendor_tier", observed = True).agg(
    n_tins = ("prov_tin", "count"),
    n_cadence = ("is_cadence", "sum"),
    avg_score = ("vendor_score", "mean"),
).reset_index()

print("Tier distribution:")
print(tier_summary.to_string(index = False))

Tier distribution:
vendor_tier  n_tins  n_cadence  avg_score
    Minimal    5077          1   0.182038
        Low    1088          6   0.366259
     Medium     726          6   0.477732
       High     363        107   0.732155


In [18]:
# export
output_dir = r"C:\Users\knguy139\Documents\Projects\Data\Output"

export_cols = id_cols + all_features + ["vendor_score", "vendor_tier", "top_contributing_features"]
df[export_cols].to_csv(f"{output_dir}\\kn_rpm_cadence_scored.csv", index = False)
print(f"Exported {len(df)} rows to kn_rpm_cadence_scored.csv")

baseline_export = baseline.merge(
    importance[["feature", "lr_coef_mean", "lr_rank", "rf_importance", "rf_rank"]],
    on = "feature", how = "left"
)
baseline_export.to_csv(f"{output_dir}\\kn_rpm_cadence_feature_summary.csv", index = False)
print(f"Exported {len(baseline_export)} features to kn_rpm_cadence_feature_summary.csv")

print("\nTop 20 non-Cadence TINs by vendor_score:")
top_non_cadence = (
    df[df["is_cadence"] == 0]
    .nlargest(20, "vendor_score")
    [["prov_tin", "sample_full_nm", "total_mbrs", "vendor_score", "vendor_tier", "top_contributing_features"]]
)
print(top_non_cadence.to_string(index = False))

Exported 7254 rows to kn_rpm_cadence_scored.csv
Exported 38 features to kn_rpm_cadence_feature_summary.csv

Top 20 non-Cadence TINs by vendor_score:
 prov_tin                     sample_full_nm  total_mbrs  vendor_score vendor_tier                            top_contributing_features
208445687                ATASSI, M.D., FALEH          45      0.987122        High pct_device_lines, distinct_srvc_npis, pct_mgmt_lines
452535719               AHMED, M.D., SYED M.         388      0.985739        High distinct_srvc_npis, pct_device_lines, pct_mgmt_lines
810745237              AL-KHAFAJI, KHALID A.          81      0.983720        High         distinct_srvc_npis, pct_hf, pct_device_lines
621681750      ANG POLAND, M.D., MICHELLE C.          54      0.983115        High distinct_srvc_npis, pct_device_lines, pct_mgmt_lines
264214938              ANDERS, JACQUELINE N.         162      0.974745        High         pct_hf, distinct_srvc_npis, pct_device_lines
760423386     AGUIRRE-BURGOS, M.D.,

In [19]:
# checkpoint
import pickle

checkpoint = {
    "df": df,
    "X": X,
    "y": y,
    "model_features": model_features,
    "scaler": scaler,
    "lr_vendor_score": lr_vendor_score,
    "rf_vendor_score": rf_vendor_score,
    "baseline": baseline,
    "importance": importance,
    "best_model": best_model,
}

with open(r"C:\Users\knguy139\Documents\Projects\RPM\rpm_cadence_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)
print("Checkpoint saved.")

engine.dispose()
print("Done.")

Checkpoint saved.
Done.
